In [1]:
import json

import pandas as pd

# Export results in the Challenge's required format

The [challenge](https://data.phmsociety.org/phm2024-conference-data-challenge/) requires the predictions to be structured as follows.
```json
{
  "0": {
    "class": 1,
    "class_conf": 0.5,
    "pdf_type": "norm",
    "pdf_args": {
      "loc": -1,
      "scale": 0.1
    }
  },
  ...
}
```


In [2]:
valid_y_regr = pd.read_csv('../2-torque_target_probabilistic_regression/predictions_valid.csv')
valid_y_class = pd.read_csv('../3-fault_detection/predictions_valid.csv')
valid_y = pd.concat([valid_y_regr, valid_y_class], axis='columns')

test_y_regr = pd.read_csv('../2-torque_target_probabilistic_regression/predictions_test.csv')
test_y_class = pd.read_csv('../3-fault_detection/predictions_test.csv')
test_y = pd.concat([test_y_regr, test_y_class], axis='columns')

In [3]:
def export_to_json(df, filename):
    results = {}
    for i, (trq_margin_mu, trq_margin_std, confidence) in df.iterrows():
        faulty = confidence > 0.5
        results[i] = {
            "class": 1 if faulty else 0,
            "class_conf": confidence * 2 - 1 if faulty else 1 - confidence * 2,
            "pdf_type": "norm",
            "pdf_args": {
                "loc": trq_margin_mu,
                "scale": trq_margin_std
            }
        }
    with open(f'{filename}.json', 'w') as fp:
        json.dump(results, fp)

In [4]:
# 2s
export_to_json(valid_y, 'validation_submission')
export_to_json(test_y, 'submission')